# Camera and Network Diagnostics

Board-side camera, DepthNet, can detector, and AprilTag diagnostics using the same services as the production runtime.

In [ ]:
from __future__ import print_function

import json
import os
import sys
import threading
import time
import traceback

def find_project_root():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isfile(os.path.join(current, 'config.json')):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise RuntimeError('config.json not found')

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import cv2
import ipywidgets as widgets
from IPython.display import Javascript, display

from demo_core import DemoDiagnostics, load_config
from demo_core.config import DIAGNOSTIC_OUTPUT_DIR

diagnostics = None
live_running = False
live_thread = None
output = widgets.Output(layout={'border': '1px solid #ccc', 'height': '430px', 'overflow_y': 'auto'})
image = widgets.Image(format='jpeg', width=320, height=240)
reload_models = widgets.Checkbox(value=False, description='reload_models')
live_fps = widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.2, description='live_fps')

def scroll_log():
    display(Javascript("setTimeout(function(){var x=document.querySelectorAll('.widget-output');for(var i=0;i<x.length;i++){x[i].scrollTop=x[i].scrollHeight;}},80);"))

def logged(fn):
    def wrapped(_=None):
        with output:
            try:
                print('\n>>> {}'.format(fn.__name__))
                print('[result] {}'.format(fn()))
            except Exception as exc:
                print('[error] {}'.format(exc))
                traceback.print_exc()
            scroll_log()
    return wrapped

def reload_config():
    global diagnostics
    if diagnostics is not None:
        diagnostics.release_camera()
    config = load_config(overrides={'runtime': {'dry_run': {'camera': False, 'base': True, 'arm': True}}})
    diagnostics = DemoDiagnostics(config)
    return 'config reloaded'

def current():
    if diagnostics is None:
        reload_config()
    return diagnostics

def start_camera():
    frame = current().start_camera()
    return None if frame is None else {'shape': frame.shape, 'dtype': str(frame.dtype)}

def capture_photo():
    frame = current().read_frame()
    if frame is None:
        raise RuntimeError('camera has no frame')
    if not os.path.isdir(str(DIAGNOSTIC_OUTPUT_DIR)):
        os.makedirs(str(DIAGNOSTIC_OUTPUT_DIR))
    path = os.path.join(str(DIAGNOSTIC_OUTPUT_DIR), 'camera_{}.jpg'.format(int(time.time())))
    cv2.imwrite(path, frame)
    return path

def load_all():
    current().start_camera()
    return current().load_all(bool(reload_models.value))

def test_depth():
    current().start_camera()
    return current().observe_depth()

def test_can():
    current().start_camera()
    current().load_can(bool(reload_models.value))
    return current().observe_can()

def test_tag():
    current().start_camera()
    current().load_tag(bool(reload_models.value))
    return current().observe_tag()

def preflight():
    return current().preflight(bool(reload_models.value))

def draw_detection(frame, result, color, label):
    if not result or not result.get('found'):
        return
    box = result.get('bbox')
    if box:
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    cx = int(result.get('center_x', 0)); cy = int(result.get('center_y', 0))
    cv2.circle(frame, (cx, cy), 5, color, -1)
    cv2.putText(frame, '{} err={} h={}'.format(label, result.get('error_x'), result.get('bbox_height_norm')), (8, 22 if label == 'can' else 44), cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1)

def live_loop():
    global live_running
    while live_running:
        try:
            frame = current().read_frame()
            if frame is not None:
                overlay = frame.copy()
                h, w = overlay.shape[:2]
                cv2.line(overlay, (w // 2, 0), (w // 2, h - 1), (255, 255, 255), 1)
                if current().services.can_detector.net is not None:
                    draw_detection(overlay, current().observe_can(), (0, 255, 0), 'can')
                tag = current().services.bin_detector
                if tag.detector is not None or tag.aruco_dict is not None:
                    draw_detection(overlay, current().observe_tag(), (0, 0, 255), 'tag')
                ok, encoded = cv2.imencode('.jpg', overlay)
                if ok:
                    image.value = encoded.tobytes()
        except Exception as exc:
            with output:
                print('[live] {}'.format(exc))
        time.sleep(1.0 / max(0.2, float(live_fps.value)))

def start_live():
    global live_running, live_thread
    start_camera()
    if live_running:
        return 'already running'
    live_running = True
    live_thread = threading.Thread(target=live_loop)
    live_thread.daemon = True
    live_thread.start()
    return 'live started'

def stop_live():
    global live_running, live_thread
    live_running = False
    if live_thread is not None:
        live_thread.join(1.5)
    live_thread = None
    return 'live stopped'

def release_camera():
    stop_live()
    return current().release_camera()

def reset_models():
    return current().reset_models()

def clear_log():
    output.clear_output()
    return True

items = [('Reload Config', reload_config), ('Start Camera', start_camera), ('Capture', capture_photo), ('Load All', load_all), ('Test Depth', test_depth), ('Test Can', test_can), ('Test Tag', test_tag), ('Preflight', preflight), ('Start Live', start_live), ('Stop Live', stop_live), ('Release Camera', release_camera), ('Reset Models', reset_models), ('Clear Log', clear_log)]
buttons = []
for label, fn in items:
    button = widgets.Button(description=label, layout=widgets.Layout(width='145px'))
    button.on_click(logged(fn))
    buttons.append(button)
display(widgets.VBox([widgets.HBox([reload_models, live_fps]), widgets.HBox(buttons[0:4]), widgets.HBox(buttons[4:8]), widgets.HBox(buttons[8:13]), image, output]))
print('[diagnostics] ready')
